**Purpose:** Comparison against state-of-the-art references.

**Inputs:** `01_data/aux/DGS1MO_maxdata.csv`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


In [98]:
import pandas as pd
import yfinance as yf
import numpy as np

class MarketMetrics:
    def __init__(self, prices: np.ndarray | list, periods_per_year: int = 252):
        """
        Args:
            prices: 1D array/list of ETF close prices
            periods_per_year: 252 for daily, 12 for monthly, etc.
        """
        self.ppy = 252
        # self.prices = np.asarray(prices, dtype=float)
        self.returns = prices.pct_change() #np.diff(self.prices) / self.prices[:-1]  # simple returns
        

        self.rf = (pd.read_csv(str(PROJECT_ROOT / "01_data/aux/DGS1MO_maxdata.csv"))
           .set_index("observation_date")
        #    .pipe(lambda df: df.loc["2023-07-01":])
           .rename_axis(None)
           .pipe(lambda s: pd.to_datetime(s.index).to_series(index=s.index).pipe(lambda _: s.set_axis(pd.to_datetime(s.index))))
           .reindex(self.returns.index.tz_localize(None))
           .ffill()
           .bfill()).values / 100
        
        self.returns = self.returns.iloc[1:]  # drop first NaN

    def cumulative_return(self) -> float:
        return np.prod(1 + self.returns) - 1

    def volatility(self) -> float:
        return np.std(self.returns) * np.sqrt(self.ppy)

    def sharpe_ratio(self) -> float:
        """risk_free_rate: annualised (e.g. 0.05 for 5%)"""

        # check for nans
        print("rf", self.rf.shape, np.isnan(self.rf).sum())
        print("returns", self.returns.shape, np.isnan(self.returns).sum())
        excess = self.returns.values - self.rf / self.ppy
        return np.mean(excess) / (np.std(excess) + 1e-8) * np.sqrt(self.ppy)

    def max_drawdown(self) -> float:
        cum = np.cumprod(1 + self.returns)
        peak = np.maximum.accumulate(cum)
        dd = (cum - peak) / peak
        return dd.min()
    
    def annualized_return(self) -> float:
        n_periods = len(self.returns)
        total_return = np.prod(1 + self.returns)
        return total_return ** (self.ppy / n_periods) - 1

    def summary(self) -> dict:
        return {
            "annualized_return":  self.annualized_return(),
            "cumulative_return": self.cumulative_return(),
            "volatility":        self.volatility(),
            "sharpe_ratio":      self.sharpe_ratio(),
            "max_drawdown":      self.max_drawdown(),
        }

In [99]:
prices = yf.download("^GSPC", start="2020-01-01", end="2024-12-31")["Close"].squeeze()

print(prices.head()), print(prices.tail())

m = MarketMetrics(prices)
m.summary()

[*********************100%***********************]  1 of 1 completed

Date
2020-01-02    3257.850098
2020-01-03    3234.850098
2020-01-06    3246.280029
2020-01-07    3237.179932
2020-01-08    3253.050049
Name: ^GSPC, dtype: float64
Date
2024-12-23    5974.069824
2024-12-24    6040.040039
2024-12-26    6037.589844
2024-12-27    5970.839844
2024-12-30    5906.939941
Name: ^GSPC, dtype: float64
rf (1257, 1) 0
returns (1256,) 0


{'annualized_return': np.float64(0.12681053205044623),
 'cumulative_return': np.float64(0.8131404958306088),
 'volatility': np.float64(0.2133881772595763),
 'sharpe_ratio': np.float64(0.5502072069653668),
 'max_drawdown': np.float64(-0.3392496000265329)}

choosing best model per dataset per paper (best model, withing the tested/focused models); only if comparable to the sp500; and if model improves its own benchmarks; only some studies, most relevant

postive var = improved, negative var = worsened, compared to the benchmark (sp500)

# IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES

- chosen most relevant papers, which use assets withing the US markets so its comparable to sp 500

- for each paper, chosen the best model from the focused ones, and only if it improves the benchmarks of the paper itself

- some papers report the Market metrics, some do not. for those, we computed the metrics using our formulas and 1M-TBills for the risk free rate, for the sp500, in the same time period. This assumption for the risk free might biase the results since their risk free rate might be lower, overestimating the Sharpe Ratio

- also assets universe is different, so its not a perfect comparison

---

- postive variations are improvements, negative variations are worsened, compared to the benchmark (sp500)

# IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES IMPORTANT NOTES


- de270 : may 2017 to may 2020 : sp500

    - news: AR 19.54, MDD 33.72, SR 0.77

{'annualized_return': np.float64(0.08210704087193577),
 'cumulative_return': np.float64(0.27466051969823946),
 'volatility': np.float64(0.21710123086276945),
 'sharpe_ratio': np.float64(0.39951606915500154),
 'max_drawdown': np.float64(-0.3392496000265328)}

In [100]:
# de270, news, 2017-2020
{'ar': 138.00243605359316,
 'vol': None,
 'mdd': 0.60134418111072,
 'sr': 92.74092615769712}

ar = (19.54 - 8.21) / 8.21 * 100
mdd = -(33.72 - 33.924) / 33.924 * 100
sr = (0.77 - 0.3995) / 0.3995 * 100

{
    "ar": ar,
    "vol": None,
    "mdd": mdd,
    "sr": sr,
}

{'ar': 138.00243605359316,
 'vol': None,
 'mdd': 0.60134418111072,
 'sr': 92.74092615769712}

- 0df79 : 2012−2021 averaged into 1 year : sp 500

    - tech: AR 0.1211 SHARPE 1.1662 MDD -0.3296

{'annualized_return': np.float64(0.14136369333717758),
 'cumulative_return': np.float64(2.7419774804725394),
 'volatility': np.float64(0.16349486532512608),
 'sharpe_ratio': np.float64(0.8566207353725043),
 'max_drawdown': np.float64(-0.33924960002653287)}

In [101]:
# 0df79, tech, 2012-2021
{'ar': -14.334439670336755,
 'vol': None,
 'mdd': 2.844395402611578,
 'sr': 36.13959502076191}

ar = (0.1211 - 0.14136369333717758) / 0.14136369333717758 * 100
mdd = -(0.3296 - 0.33924960002653287) / 0.33924960002653287 * 100
sr = (1.1662 - 0.8566207353725043) / 0.8566207353725043 * 100

{
    "ar": ar,
    "vol": None,
    "mdd": mdd,
    "sr": sr,
}

{'ar': -14.334439670336755,
 'vol': None,
 'mdd': 2.844395402611578,
 'sr': 36.13959502076191}

- 83780 : 1989 - 2013 : sp 500

    - MARKET (bench): Return 10.39, VOL 15.14, SHARPE 0.46
    - tech and macro (shorting allowed): Return 14.45, VOL 11.42, SHARPE 0.97

{}

In [102]:
# 8370, tech and macro, 1989-2013
{'ar': 39.076034648700656,
 'vol': 24.57067371202114,
 'mdd': None,
 'sr': 110.86956521739131}

ar = (14.45 - 10.39) / 10.39 * 100
vol = -(11.42 - 15.14) / 15.14 * 100
sr = (0.97 - 0.46) / 0.46 * 100

{
    "ar": ar,
    "vol": vol,
    "mdd": None,
    "sr": sr,
}

{'ar': 39.076034648700656,
 'vol': 24.57067371202114,
 'mdd': None,
 'sr': 110.86956521739131}

- e6ca3 : 12fev2024 a 30mai2025: market is sp500

    - tech: AR 25.10, VOL 26.29, SHARPE 0.95, MDD -23.35
    - news: AR 31.23, VOL 27.19, SHARPE 1.15, MDD -22.05

{'annualized_return': np.float64(0.13535594541453433),
 'cumulative_return': np.float64(0.17729161140673955),
 'volatility': np.float64(0.18220803801739466),
 'sharpe_ratio': np.float64(0.5149388573831384),
 'max_drawdown': np.float64(-0.1890220618428394)}

In [103]:
# e6ca3, tech, 2024-2025
{'ar': 85.43699667665132,
 'vol': -44.28562145809508,
 'mdd': -23.530553906528315,
 'sr': 84.48792247448436}

ar = (0.2510 - 0.13535594541453433) / 0.13535594541453433 * 100
mdd = -(0.2335 - 0.1890220618428394) / 0.1890220618428394 * 100
sr = (0.95 - 0.5149388573831384) / 0.5149388573831384 * 100
vol = -(0.2629 - 0.18220803801739466) / 0.18220803801739466 * 100

{
    "ar": ar,
    "vol": vol,
    "mdd": mdd,
    "sr": sr,
}

{'ar': 85.43699667665132,
 'vol': -44.28562145809508,
 'mdd': -23.530553906528315,
 'sr': 84.48792247448436}

In [104]:
# e6ca3, news, 2024-2025
{'ar': 130.72499626341917,
 'vol': -49.225030332658974,
 'mdd': -16.653049834644502,
 'sr': 123.3274851006916}

ar = (0.3123 - 0.13535594541453433) / 0.13535594541453433 * 100
mdd = -(0.2205 - 0.1890220618428394) / 0.1890220618428394 * 100
sr = (1.15 - 0.5149388573831384) / 0.5149388573831384 * 100
vol = -(0.2719 - 0.18220803801739466) / 0.18220803801739466 * 100

{
    "ar": ar,
    "vol": vol,
    "mdd": mdd,
    "sr": sr,
}

{'ar': 130.72499626341917,
 'vol': -49.225030332658974,
 'mdd': -16.653049834644502,
 'sr': 123.3274851006916}

- 4e332: 2020 a 2024 : -
    - macro+tech: AR 18.9, SR 1.17, MDD -29.3
    - MARKET (bench): AR 11.7, SR 0.66, MDD -41.2

{}

In [105]:
# 4e332, tech and macro, 2020-2024
{'ar': 61.53846153846153,
 'vol': None,
 'mdd': 28.883495145631073,
 'sr': 77.27272727272725}

ar = (18.9 - 11.7) / 11.7 * 100
mdd = -(29.3 - 41.2) / 41.2 * 100
sr = (1.17 - 0.66) / 0.66 * 100

{
    "ar": ar,
    "vol": None,
    "mdd": mdd,
    "sr": sr,
}

{'ar': 61.53846153846153,
 'vol': None,
 'mdd': 28.883495145631073,
 'sr': 77.27272727272725}

- 984be: 2011 to the end of April 2020 : sp500
    - tech: R 0.313, SR 1.858, MDD 0.102 VOL 0.168

{'annualized_return': np.float64(0.09420422654573168),
 'cumulative_return': np.float64(1.3111717558010123),
 'volatility': np.float64(0.17299600495376172),
 'sharpe_ratio': np.float64(0.5728760874419057),
 'max_drawdown': np.float64(-0.3392496000265329)}

In [106]:
# 984be, tech, 2011-2020
{'ar': 232.25685457759516,
 'vol': 2.887930825395093,
 'mdd': 69.93364178114801,
 'sr': 224.32842646594437}

ar = (0.313 - 0.09420422654573168) / 0.09420422654573168 * 100
mdd = -(0.102 - 0.3392496000265329) / 0.3392496000265329 * 100
sr = (1.858 - 0.5728760874419057) / 0.5728760874419057 * 100
vol = -(0.168 - 0.17299600495376172) / 0.17299600495376172 * 100

{
    "ar": ar,
    "vol": vol,
    "mdd": mdd,
    "sr": sr,
}

{'ar': 232.25685457759516,
 'vol': 2.887930825395093,
 'mdd': 69.93364178114801,
 'sr': 224.32842646594437}

In [107]:
my_results = """ac{sp500}  & $0.394$ & $0.803$ & $0.161$ & $-0.189$ \\
EW benchmark & $0.315$ & $0.704$ & $0.137$ & $-0.153$ \\
MVO-CAMP benchmark & $0.380$ & $0.835$ & $0.146$ & $-0.188$ \\

LSTM via returns technical & $0.347$ & $0.765$ & $0.143$ & $-0.149$ \\
LSTM via sharpe technical & $0.471$ & $0.829$ & $0.197$ & $-0.238$ \\
DRL PPO technical & $0.371$ & $0.809$ & $0.147$ & $-0.152$ \\

LSTM via returns news & $0.360$ & $0.817$ & $0.139$ & $-0.159$ \\
LSTM via sharpe news & $0.447$ & $0.907$ & $0.164$ & $-0.149$ \\
DRL PPO news & $0.424$ & $0.976$ & $0.140$ & $-0.116$ \\

LSTM via returns reddit & $0.481$ & $1.164$ & $0.134$ & $-0.097$ \\
LSTM via sharpe reddit & $0.855$ & $1.212$ & $0.238$ & $-0.257$ \\
DRL PPO reddit & $0.538$ & $1.173$ & $0.151$ & $-0.147$ \\

LSTM via returns reddit+news & $0.416$ & $0.967$ & $0.138$ & $-0.117$ \\
LSTM via sharpe reddit+news & $0.580$ & $1.081$ & $0.181$ & $-0.161$ \\
DRL PPO reddit+news & $0.596$ & $1.222$ & $0.162$ & $-0.130$ \\"""

rows = my_results.split("\\\n")

def annualized_return(cumulative_return, days):
    return (1 + cumulative_return) ** (365 / days) - 1

# Example: 50% return over 2 years (~730 days)
days = 499 #730

results = {}
market = {}
for row in rows:
    cols = row.split("&")
    name = cols[0].strip()
    CUMRET = float(cols[1].strip().strip("$"))
    SHARPE = float(cols[2].strip().strip("$"))
    VOL = float(cols[3].strip().strip("$"))
    MDD = float(cols[4].strip().strip("$").strip("\\").strip(" ").strip("$"))

    if name == "ac{sp500}":
        market = {
            "AR": annualized_return(CUMRET, days),
            "SHARPE": SHARPE,
            "VOL": VOL,
            "MDD": MDD,
        }

    else:
        results[name] = {
            "AR": annualized_return(CUMRET, days),
            "SHARPE": SHARPE,
            "VOL": VOL,
            "MDD": MDD,
        }

In [108]:
for name, metrics in results.items():
    print(f"{name}:")

    adj_metrics = {
        "AR": (metrics["AR"] - market["AR"]) / market["AR"] * 100,
        "SHARPE": (metrics["SHARPE"] - market["SHARPE"]) / market["SHARPE"] * 100,
        "VOL": -(metrics["VOL"] - market["VOL"]) / market["VOL"] * 100,
        "MDD": -(metrics["MDD"] - market["MDD"]) / market["MDD"] * 100,
    }
    print(adj_metrics)

EW benchmark:
{'AR': -19.366954346576094, 'SHARPE': -12.328767123287681, 'VOL': 14.90683229813664, 'MDD': 19.04761904761905}
MVO-CAMP benchmark:
{'AR': -3.410175097048479, 'SHARPE': 3.98505603985055, 'VOL': 9.316770186335411, 'MDD': 0.5291005291005296}
LSTM via returns technical:
{'AR': -11.485471368357851, 'SHARPE': -4.732254047322544, 'VOL': 11.180124223602494, 'MDD': 21.164021164021168}
LSTM via sharpe technical:
{'AR': 18.594827617174342, 'SHARPE': 3.237858032378569, 'VOL': -22.360248447204974, 'MDD': -25.92592592592592}
DRL PPO technical:
{'AR': -5.607335595186772, 'SHARPE': 0.7471980074719807, 'VOL': 8.695652173913052, 'MDD': 19.57671957671958}
LSTM via returns news:
{'AR': -8.298021841274407, 'SHARPE': 1.7434620174346076, 'VOL': 13.66459627329192, 'MDD': 15.873015873015872}
LSTM via sharpe news:
{'AR': 12.827710198935652, 'SHARPE': 12.951432129514318, 'VOL': -1.8633540372670823, 'MDD': 21.164021164021168}
DRL PPO news:
{'AR': 7.276738010153188, 'SHARPE': 21.54420921544208, 'VOL'